In [1]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

# 1. Tải các biến môi trường từ file .env
load_dotenv()

# 2. Lấy API key
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Chưa thiết lập GOOGLE_API_KEY trong file .env")

# 3. Cấu hình API
genai.configure(api_key=api_key)

# 4. Khởi tạo mô hình

model = genai.GenerativeModel('gemini-2.5-flash')

# # 5. Gửi yêu cầu (prompt)
# prompt = "Viết một bài thơ 4 câu về thành phố Hồ Chí Minh"
# response = model.generate_content(prompt)
#
# # 6. In kết quả
# print(response.text)

D:\uit_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
summarize_text_prompt = """VAI TRÒ & MỤC TIÊU: Bạn là một AI chuyên gia trích xuất thông tin. Nhiệm vụ của bạn là nhận một văn bản thô từ người dùng và "làm phẳng" (flatten) nó thành các sự kiện cốt lõi để chuẩn bị cho tác vụ Tách Triplet (Triplet Extraction).

MỤC TIÊU CHÍNH: Đầu ra KHÔNG cần phải có ngữ nghĩa mượt mà cho người đọc. Mục tiêu là tạo ra một chuỗi dữ liệu mà máy tính có thể dễ dàng phân tích thành các bộ (Chủ thể - Quan hệ - Tân ngữ).

ĐẦU VÀO (INPUT): {text_input}

QUY TẮC BẮT BUỘC:

1.  **Loại bỏ "Nhiễu" (Noise Reduction):**
    * Loại bỏ tuyệt đối các từ đệm, từ thừa không mang nghĩa (ví dụ: thì, là, mà, à, ừm).
    * Loại bỏ các cụm từ mang tính hội thoại, cảm xúc, hoặc xin phép (ví dụ: "tôi lo lắng", "xin vui lòng", "bạn có thể cho tôi biết", "giúp tôi với").

2.  **Giữ lại "Tín Hiệu" (Signal Retention):**
    * Tập trung giữ lại các **thực thể chính** (ai, cái gì, ở đâu).
    * Tập trung giữ lại các **hành động/quan hệ** (làm gì, như thế nào).
    * Giữ lại các **điều kiện** quan trọng (thời gian, địa điểm, điều kiện nếu-thì).

3.  **Đảm bảo Chất lượng Đầu ra (Output Constraints):**
    * **Ngắn gọn:** Đầu ra phải ngắn hơn đáng kể so với đầu vào.
    * **Giảm độ phức tạp:** Chuyển các cấu trúc câu phức, rườm rà thành các cụm từ khóa hoặc câu đơn súc tích nhất có thể.
    * **Giảm nhiễu ngữ nghĩa:** Loại bỏ mọi chi tiết phụ không liên quan trực tiếp đến ý chính của câu hỏi.


4.  **PHÂN BIỆT BỐI CẢNH VÀ NỘI DUNG HỎI (QUAN TRỌNG NHẤT):**
    * Bạn **CHỈ ĐƯỢC** tóm tắt các sự kiện, bối cảnh, tình huống được cung cấp trong câu.
    * Bạn **PHẢI LOẠI BỎ** hoàn toàn nội dung, chủ đề, hoặc hành động đang được hỏi. **Tuyệt đối không** được biến phần câu hỏi thành một câu trần thuật/khẳng định.
---
QUY TRINH THỰC HIỆN: Hãy phân tích văn bản đầu vào và áp dụng các quy tắc sau một cách nghiêm ngặt:

PHẦN 1: QUY TẮC LỌC NỘI DUNG (GIỮ LẠI vs. LOẠI BỎ)

Bạn PHẢI LOẠI BỎ tất cả những điều sau:

Cảm xúc & Tính chủ quan: Bất kỳ từ ngữ nào thể hiện tâm trạng (tôi rất lo lắng, bức xúc, quá căng thẳng, tôi nghĩ là).

Đại từ nhân xưng: Tất cả các đại từ (tôi, chúng tôi, anh ấy, công ty họ). Hãy thay thế bằng vai trò pháp lý của họ (người lao động, người mua, bên A).

Thông tin cá nhân không liên quan: Tên riêng cụ thể, địa chỉ nhà, số điện thoại.

Từ ngữ đệm & Lặp lại: Các từ thừa, không mang ý nghĩa pháp lý (thì, là, mà, vấn đề là, chuyện là).

Bạn PHẢI GIỮ LẠI VÀ CHUẨN HÓA tất cả những điều sau:

Thời gian & Điều kiện (Các Thuộc tính):

Thời hạn/Mốc thời gian: 30 ngày, sau 2 năm, kể từ ngày 1/1/2024.

Điều kiện & Ngoại lệ: nếu không thông báo trước, trừ trường hợp bất khả kháng, khi tài sản bị hư hỏng.

PHẦN 2: QUY TẮC TÁI CẤU TRÚC CÂU

Mục tiêu là đơn giản hóa ngữ pháp để máy có thể dễ dàng phân tích quan hệ.

Chuyển đổi câu hỏi: KHÔNG tóm tắt câu hỏi. Thay vào đó, hãy tóm tắt sự kiện dẫn đến câu hỏi đó.

Đơn giản hóa câu: Chuyển đổi cấu trúc bị động thành chủ động khi có thể (ví dụ: "Người lao động bị công ty sa thải" -> "Công ty sa thải người lao động").

PHẦN 3: QUY TẮC ĐỊNH DẠNG ĐẦU RA (NGHIÊM NGẶT)

Đây là quy tắc bắt buộc để đảm bảo tính độc lập của dữ liệu cho ERE.

Chỉ sử dụng câu đơn: Toàn bộ đầu ra phải được chia thành các câu đơn. Mỗi câu chỉ mô tả một sự kiện, một mối quan hệ, hoặc một thuộc tính. Tuyệt đối không dùng câu ghép, câu phức.

Độc lập về ngữ nghĩa: Mỗi câu đơn phải hoàn toàn độc lập về mặt ý nghĩa. Người đọc (hoặc máy) phải hiểu được câu đó mà không cần đọc câu trước hoặc câu sau.

Không tham chiếu chéo: Tránh sử dụng đại từ (họ, nó, anh ta) hoặc các cụm từ tham chiếu (việc này, sau đó) để liên kết với các câu trước. Nếu cần, hãy lặp lại chủ thể một cách rõ ràng.


Ví dụ (SAI): "Bên A ký hợp đồng. Họ chưa thanh toán."

Ví dụ (ĐÚNG): "Bên A ký hợp đồng. Bên A chưa thanh toán."

Phân tách bằng dấu chấm: Mỗi câu đơn phải kết thúc bằng một dấu chấm (.). Các câu được đặt liền nhau, chỉ ngăn cách bởi dấu chấm và một khoảng trắng.


---
BẢN TÓM TẮT BỐI CẢNH (OUTPUT):
(Chỉ cung cấp văn bản tóm tắt đã được làm sạch theo các quy tắc 1, 2, và 3)"""

In [3]:
def summarize_text(text_input):
    prompt = summarize_text_prompt.format(text_input=text_input)
    summarized_text = model.generate_content(prompt).text
    summarized_text
    return summarized_text

In [6]:
text_input = "Tôi lái xe máy vượt đèn đỏ, tôi sẽ bị phạt như thế nào?"

In [7]:
summarize_text(text_input)

'Người lái xe máy vượt đèn đỏ.'